# 04 · Hocanın istediği iki ek analiz
GPU gerekmez. Yeni model eğitilmez, yeni skor hesaplanmaz. `02_tum_deneyler` notebook'unun Drive'a kaydettiği skorları ve kare dosyalarının adlarını okur. Birkaç dakikada biter.

- **Bölüm 1 (Madde 2):** UCF-Crime karelerinin her videoda hangi kare aralığından geldiği. Resmi zamansal etiket dosyası Drive'da varsa, "anomali" diye kullanılan karelerin kaçının gerçekten olay aralığına düştüğü de hesaplanır.
- **Bölüm 2 (Madde 3):** Avenue'da model ile öğrenmesiz skorlar arasındaki AUC farkı için video düzeyinde eşleştirilmiş bootstrap güven aralığı.

Çıktılar: `revizyon_sonuclari_v2/ek_analizler/`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, re, glob, json
import numpy as np, pandas as pd
from scipy.io import loadmat
from sklearn.metrics import roc_auc_score

PROJ    = '/content/drive/MyDrive/AnomalyProject'
FRAMES  = f'{PROJ}/frames'
OUT_DIR = f'{PROJ}/revizyon_sonuclari_v2'
EK      = f'{OUT_DIR}/ek_analizler'
AVENUE_GT = f'{PROJ}/avenue_gt/testing_label_mask'
# İsteğe bağlı: UCF-Crime'ın resmi zamansal etiket dosyası. Varsa bu yola koyun, yoksa Bölüm 1 yine çalışır.
UCF_ANN = f'{PROJ}/data/Temporal_Anomaly_Annotation_for_Testing_Videos.txt'
# Klasör adları eski; içerik dosya adlarından doğrulandı (02 notebook'u ile aynı).
UCF = {'Normal': 'ucf_normal', 'Kavga (Fighting)': 'ucf_fighting',
       'Hırsızlık (Stealing)': 'ucf_robbery', 'Soygun (Burglary)': 'ucf_accident'}
N_BOOT, SEED = 10000, 42
os.makedirs(EK, exist_ok=True)

## Bölüm 1 · UCF-Crime kareleri hangi aralıktan alınmış? (Madde 2)
Kare dosyalarının adı `VideoAdı_x264_<sayı>.<uzantı>` biçiminde. Sondaki sayı karenin videodaki sırasıdır. Aşağıdaki tablo her video için kare sayısını, ilk ve son sayıyı ve iki kare arasındaki adımı verir.

**Adım sütununu okuyun:**
- Adım 1 ise sayılar 0, 1, 2, … diye gidiyor. Bu bir sayaçtır, videonun orijinal kare numarası değildir. Kareler videodan belli aralıklarla (örneğin her 10 karede bir) çıkarılmış olabilir. Orijinal kare numarası bu durumda "sayı × çıkarım aralığı" olur.
- Adım 10 gibi bir değerse sayılar doğrudan orijinal kare numarasıdır.

In [ ]:
pat = re.compile(r'^(?P<vid>.+?)_(?P<i>\d+)\.(?:png|jpe?g|bmp)$', re.I)
rows = []
for cat, folder in UCF.items():
    files = sorted(os.listdir(f'{FRAMES}/{folder}'))
    by = {}
    for f in files:
        m = pat.match(f)
        if m: by.setdefault(m['vid'], []).append(int(m['i']))
    for vid, ix in by.items():
        ix = np.sort(ix); d = np.diff(ix)
        rows.append({'Kategori': cat, 'Klasör': folder, 'Video': vid, 'Kare sayısı': len(ix),
                     'İlk sayı': int(ix[0]), 'Son sayı': int(ix[-1]),
                     'Adım (medyan)': int(np.median(d)) if len(d) else 0,
                     'Boşluk var mı': 'evet' if len(d) and (d != np.median(d)).any() else 'hayır'})
    unmatched = [f for f in files if not pat.match(f)]
    if unmatched: print(f'⚠ {folder}: adı beklenen biçimde olmayan {len(unmatched)} dosya, ör. {unmatched[:3]}')
KA = pd.DataFrame(rows)
KA.to_csv(f'{EK}/ucf_kare_araliklari.csv', index=False, encoding='utf-8-sig')
display(KA)
print('Toplam:', KA['Kare sayısı'].sum(), 'kare,', len(KA), 'video')

### İsteğe bağlı · Resmi zamansal etiketlerle karşılaştırma
UCF-Crime'ın resmi test videoları için `Temporal_Anomaly_Annotation_for_Testing_Videos.txt` dosyası olayın başladığı ve bittiği kare numaralarını verir. Dosya, veri setinin resmi indirmesindeki `UCF_Crimes-Train-Test-Split` klasöründe bulunur. Drive'a yukarıdaki `UCF_ANN` yoluna koyarsanız bu hücre şunları yapar:
- Videolarımızın resmi test listesinde olup olmadığını gösterir.
- Anomali videolarından alınan karelerin kaçının olay aralığının içine, kaçının dışına düştüğünü hesaplar.

Kare sayısını orijinal kare numarasına çevirmek için çıkarım aralığı gerekir. Bunu `EXTRACT_STEP` ile verin: adım sütunu 1 çıktıysa ve kareler her 10 karede bir alınmışsa 10, adım sütunu zaten 10 gibi bir değerse 1.

In [ ]:
EXTRACT_STEP = 10   # yukarıdaki açıklamaya göre değiştirin
if os.path.exists(UCF_ANN):
    ann = {}
    for line in open(UCF_ANN, encoding='utf-8', errors='ignore'):
        p = line.split()
        if len(p) >= 6: ann[p[0].replace('.mp4', '')] = [(int(p[2]), int(p[3])), (int(p[4]), int(p[5]))]
    out = []
    for cat, folder in UCF.items():
        for f in os.listdir(f'{FRAMES}/{folder}'):
            m = pat.match(f)
            if not m: continue
            vid, fr = m['vid'], int(m['i']) * EXTRACT_STEP
            iv = ann.get(vid)
            inside = None if iv is None else any(a >= 0 and a <= fr <= b for a, b in iv)
            out.append({'Kategori': cat, 'Video': vid, 'Resmi test listesinde': iv is not None, 'Olay içinde': inside})
    O = pd.DataFrame(out)
    S = O.groupby(['Kategori', 'Video']).agg(Kare=('Video', 'size'), Resmi_testte=('Resmi test listesinde', 'first'),
                                             Olay_icinde=('Olay içinde', lambda x: int(np.nansum([v is True for v in x])))).reset_index()
    S['Olay dışında'] = np.where(S['Resmi_testte'], S['Kare'] - S['Olay_icinde'], np.nan)
    for v in S['Video']:
        if v in ann: print(f'{v}: resmi olay aralıkları {[iv for iv in ann[v] if iv[0] >= 0]}')
    S.to_csv(f'{EK}/ucf_resmi_etiket_karsilastirma.csv', index=False, encoding='utf-8-sig'); display(S)
else:
    print('Resmi etiket dosyası yok, bu adım atlandı:', UCF_ANN)

## Bölüm 2 · Model ile öğrenmesiz skorlar arasındaki AUC farkı (Madde 3)
Tablo 4'te Avenue'da model %86,75, en iyi öğrenmesiz skor (Avenue ortalama kareye uzaklık) %82,61 AUC aldı. Fark yaklaşık 4 puan. Soru şu: bu fark gerçek mi, yoksa 21 test videosunun hangileri olduğuna bağlı bir şans mı?

Yöntem:
1. 21 test videosundan, iadeli olarak rastgele 21 video seçilir. Aynı video birden fazla kez seçilebilir.
2. Seçilen videolarda modelin AUC'si ile öğrenmesiz skorun AUC'si **aynı videolar üzerinden** hesaplanır ve farkları alınır. Buna "eşleştirilmiş" denmesinin nedeni bu.
3. Bu işlem 10.000 kez tekrarlanır. Farkların %2,5 ve %97,5 yüzdelikleri %95 güven aralığını verir.

Güven aralığı tamamen sıfırın üstündeyse model öğrenmesiz skordan anlamlı biçimde iyidir. Aralık sıfırı içeriyorsa fark bu test kümesinde istatistiksel olarak ayırt edilemiyor demektir.

In [ ]:
mats = sorted(glob.glob(f'{AVENUE_GT}/*_label.mat'), key=lambda s: int(re.findall(r'(\d+)_label', s)[0]))
y = []
for m in mats:
    vol = loadmat(m)['volLabel']; y.append(np.array([int(np.asarray(vol[0, i]).sum() > 0) for i in range(vol.shape[1])]))
J = json.load(open(f'{OUT_DIR}/sonuclar_tam.json', encoding='utf-8')); STRIDE = int(J['AV_STRIDE'])
S0 = np.load(f'{OUT_DIR}/skorlar/seed42.npz', allow_pickle=True)['S'].item()
BS = np.load(f'{OUT_DIR}/skorlar/baselines.npz', allow_pickle=True)['B'].item()
model = S0['avenue'][STRIDE] if STRIDE in S0['avenue'] else S0['avenue'][str(STRIDE)]
assert len(model) == len(y) == 21 and all(len(a) == len(b) for a, b in zip(model, y)), 'skor ve etiket uzunlukları uyuşmuyor'

def micro(sc, pick): return roc_auc_score(np.concatenate([y[i] for i in pick]), np.concatenate([sc[i] for i in pick]))
allv = np.arange(len(y))
rng = np.random.default_rng(SEED)
picks = []
while len(picks) < N_BOOT:
    p = rng.integers(0, len(y), len(y))
    yy = np.concatenate([y[i] for i in p])
    if 0 < yy.mean() < 1: picks.append(p)
m_boot = np.array([micro(model, p) for p in picks])

rows = []
for name, B in BS.items():
    base = B['avenue']
    d_boot = m_boot - np.array([micro(base, p) for p in picks])
    rows.append({'Karşılaştırma': f'Model − {name}',
                 'Model AUC (%)': round(100 * micro(model, allv), 2), 'Öğrenmesiz AUC (%)': round(100 * micro(base, allv), 2),
                 'Fark (puan)': round(100 * (micro(model, allv) - micro(base, allv)), 2),
                 '%95 GA alt': round(100 * np.percentile(d_boot, 2.5), 2), '%95 GA üst': round(100 * np.percentile(d_boot, 97.5), 2),
                 'Farkın ≤ 0 olduğu tekrar oranı': round(float((d_boot <= 0).mean()), 4)})
FB = pd.DataFrame(rows).sort_values('Öğrenmesiz AUC (%)', ascending=False)
FB.to_csv(f'{EK}/avenue_auc_fark_bootstrap.csv', index=False, encoding='utf-8-sig')
display(FB)

In [ ]:
f2 = lambda x: f'{x:.2f}'.replace('.', ',')
r = FB[FB['Karşılaştırma'].str.contains('ortalama kareye')].iloc[0]
lines = ['# Ek analizler (hocanın istedikleri)', '',
         '## Madde 2 · UCF-Crime kare aralıkları', '',
         KA.to_markdown(index=False), '',
         f'Toplam {KA["Kare sayısı"].sum()} kare, {len(KA)} video.', '',
         '## Madde 3 · Avenue AUC farkı, video düzeyi eşleştirilmiş bootstrap', '',
         f'{N_BOOT} tekrar, 21 test videosu iadeli örneklendi, model ve öğrenmesiz skor aynı videolar üzerinden karşılaştırıldı.', '',
         FB.to_markdown(index=False), '',
         '### Makale için önerilen cümle (değerleri kontrol edin)', '',
         f"Model ile en iyi öğrenmesiz skor (Avenue ortalama kareye uzaklık) arasındaki AUC farkı {f2(r['Fark (puan)'])} puandır "
         f"(video düzeyinde eşleştirilmiş bootstrap, {N_BOOT} tekrar, %95 GA: {f2(r['%95 GA alt'])} – {f2(r['%95 GA üst'])})."]
open(f'{EK}/ek_analiz_sayilari.md', 'w', encoding='utf-8').write('\n'.join(lines))
print('\n'.join(lines)); print('\n✅ Kaydedildi →', EK)